# Variables

This page provides an overview of tricks and tools for working with variables, including environment variables, in Linux.

---

The following cell illustrates how a variable can be defined:

In [1]:
SHELL_VARIABLE=10

The example of the invocation:

In [2]:
echo $SHELL_VARIABLE

10


Use `set` command to view current shell variables.

In [1]:
set | head -n 10

BASH=/usr/bin/bash
BASHOPTS=checkwinsize:cmdhist:complete_fullquote:expand_aliases:extglob:extquote:force_fignore:globasciiranges:globskipdots:histappend:interactive_comments:patsub_replacement:progcomp:promptvars:sourcepath
BASH_ALIASES=()
BASH_ARGC=([0]="0")
BASH_ARGV=()
BASH_CMDS=()
BASH_COMPLETION_VERSINFO=([0]="2" [1]="11")
BASH_LINENO=()
BASH_LOADABLES_PATH=/usr/local/lib/bash:/usr/lib/bash:/opt/local/lib/bash:/usr/pkg/lib/bash:/opt/pkg/lib/bash:.
BASH_SOURCE=()


## Environment

There are two types of varialbes in bash:

- **Shell variables** are accessible only within the current shell session. Any shells invoked within this shell will not inherit these variables.
- **Environment variables** are variables that are inherited by all the subsequent shells.




---

Consider `LOCAL_VARIABLE` - within this shell, you can access it freely. The following cell demonstrates how it is substituted into the output of `echo`:

In [1]:
LOCAL_VARIABLE="hello world"
echo $LOCAL_VARIABLE

hello world


However, the following cell attempts to access `LOCAL_VARIABLE` from another shell:

In [2]:
bash -c "echo \$LOCAL_VARIABLE"

There is no result — by executing `bash`, you have created another shell where `LOCAL_VARIABLE` is not defined.

The same issue with python REPL:

In [3]:
python3 -c "import os; print(os.environ[\"LOCAL_VARIABLE\"])"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.10/os.py", line 680, in __getitem__
    raise KeyError(key) from None
KeyError: 'LOCAL_VARIABLE'


: 1

However, if the variable is defined as an environment variable, the nested shells can use those values:

In [3]:
export LOCAL_VARIABLE="hello variable"
bash -c "echo \$LOCAL_VARIABLE"

export LOCAL_VARIABLE="hello variable"
python3 -c "import os; print(os.environ[\"LOCAL_VARIABLE\"])"

hello variable
hello variable


**Note**: if an environment variable is defined in a child shell, it won't be accessible from the parent shells:

In [2]:
bash -c "export ATTEMPT_TO_DEFINE_ENV=value"
echo $ATTEMPT_TO_DEFINE_ENV

### List

The `env` command in linux prints all available environment variables:

---

The following cell shows some variables from the current shell:

In [18]:
env | head -n 10

SHELL=/bin/bash
SESSION_MANAGER=local/fedor-NUC10i7FNK:@/tmp/.ICE-unix/2906,unix/fedor-NUC10i7FNK:/tmp/.ICE-unix/2906
QT_ACCESSIBILITY=1
COLORTERM=truecolor
XDG_CONFIG_DIRS=/etc/xdg/xdg-ubuntu:/etc/xdg
XDG_MENU_PREFIX=gnome-
GNOME_DESKTOP_SESSION_ID=this-is-deprecated
CONDA_EXE=/home/fedor/miniforge3/bin/conda
_CE_M=
LOCAL_VARIABLE=hello variable


### Define

The linux cmd offers the following tools to operate with environment variables:

- The `export <var_name>=<value>` to define the environment variable.
- Use the syntax `<var_name>=<var_value> <shell>` to define the variable that will be awailable in the child shell.
- Use the `set -a` option to configure the shell to export all shell variable automatically. **Note** turn off automatic exporting with the `set +a` option.

---

The `export` statement before the definition of the variable.

In [4]:
export ENVIRONMENT_VARIABLE="hello global"
echo $ENVIRONMENT_VARIABLE

hello global


Now it can be easily accessed from any nested shell.

In [5]:
python3 -c "import os; print(os.environ[\"ENVIRONMENT_VARIABLE\"])"
bash -c "echo \$ENVIRONMENT_VARIABLE"

hello global
hello global


The following cell illustrates the definition of the variable for subsequent shell. The `NESTED_VAR` is defined before the bash is invoked, meaning the nested bash can use it:

In [10]:
NESTED_VAR=value bash -c "echo \$NESTED_VAR"

value


The option to configure the `set -a` in the shell is useful when you need to load environment variables from a file (typically `.env` file). The following cell shows how to source the env file with the included `set -a` option:

In [15]:
set -a

source <(cat <<EOF
ENV_VAR1=hello
ENV_VAR2=world
EOF
)

set +a

The corresponding variables therefore appear in the output of the `env` command:

In [16]:
env | grep ENV_VAR

ENV_VAR1=hello
ENV_VAR2=world


## Keep variables in file

For convenince you can keep variables in file - typical name for such file is `.env`. In fact it is just script that have to be executed in the shell where you wan't to define varables - so you have just write command simply like that you can write in the shell, result would be the same.

To load variable from file you can use `. <file>` or `source <file>`.

---

As an example, consider a file created with the following code:

In [15]:
cat << EOF > /tmp/env
VARIABLE1="variable 1 value"
VARIABLE2="variable 2 value"
export ENVIRONMENT_EXAMPLE="environment hello"
EOF

After loading this file, you can use the variables defined in it as usual.

In [17]:
source /tmp/env
echo $VARIABLE1 $VARIABLE2
env | grep EXAMPLE
unset VARIABLE1 VARIABLE2 ENVIRONMENT_EXAMPLE

variable 1 value variable 2 value
ENVIRONMENT_EXAMPLE=environment hello


The same output with using `.` instead of `source`.

In [18]:
. /tmp/env
echo $VARIABLE1 $VARIABLE2
env | grep EXAMPLE
unset VARIABLE1 VARIABLE2 ENVIRONMENT_EXAMPLE

variable 1 value variable 2 value
ENVIRONMENT_EXAMPLE=environment hello


**Note:** In this context, you can't use `bash` to execute files that define variables, as they will only be defined in the invoked shell, not in the calling shell.

In [23]:
bash /tmp/env
echo $VARIABLE1 $VARIABLE2

## Unset variable

If you want to remove a variable from the current shell or delete a local variable, use the `unset <variable>` command.

---

The following cell creates a shell variable and an environment variable. Once created, you can use them in any command you like.

In [36]:
shell_var="hello shell"
export env_var="hello environment"

echo "$shell_var $env_var"

hello shell hello environment


But after applying `unset` to the variables, `echo` returns an empty output again.

In [37]:
unset shell_var env_var

echo "$shell_var $env_var"

## Replace env. var. (envsubst)

The `envsubst` utility is used to replace placeholders within a "template" with corresponding values from environment variables. It allows you to substitute variables in specific locations with their actual values.

---

In the following cell is created a template:

In [ ]:
cat << EOF > envsubst_example
User \${username} succesfully login his age is \${userage}.
EOF

If we print this template as it is - it will have `${username}` and `${userage}` just as text.

In [ ]:
cat envsubst_example

User ${username} succesfully login his age is ${userage}.


If we define the corresponding values `username` and `userage` and pass the file to the envsubst command, we will obtain a line with the substituted values.

In [ ]:
export username=Fedor
export userage=23
envsubst < envsubst_example
rm envsubst_example

User Fedor succesfully login his age is 23.
